In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import RetrievalQA



In [17]:
import os

In [18]:
os.environ["GOOGLE_API_KEY"] = os.getenv("api_key")

In [19]:
#Load dos modelos

embeddings_models = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview", api_key=os.getenv("api_key"))
llm = ChatGoogleGenerativeAI(model="gemini-3.1-pro-preview")

In [20]:
#carregar pdf

PDF_LINK = "1758887557527-attachment.pdf"
loader = PyPDFLoader(PDF_LINK, extract_images=False)
pages = loader.load_and_split()

#separar chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000, 
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)



In [21]:
#salvar no VECTOR DB

db = Chroma.from_documents(chunks, embedding=embeddings_models, persist_directory="text_index")



In [22]:
#carregar db

vectordb = Chroma(persist_directory="text_index", embedding_function=embeddings_models)

#load retriever

retriever = vectordb.as_retriever(search_kwargs={"k": 3})
chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

In [23]:
def ask_question(question):
    context = retriever._get_relevant_documents(question, run_manager=None)
    answer = (chain({"input_documents": context, "query": question}, return_only_outputs=True))["output_text"]
    return answer



In [24]:
user_question = input("Digite sua pergunta: ")
answer = ask_question(user_question)
print("Resposta:", answer)

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.1-pro-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-pro\nPlease retry in 33.83572228s.', 'status': 'RESOURCE_